
# E-Commerce Sales Data Cleaning

## 1. Project Overview

This notebook focuses on cleaning and preparing the raw e-commerce retail dataset for further analysis.

The original dataset contains 5,040 records and 24 columns, covering customer details, product information, sales transactions, payment methods, and order delivery information.

### Objectives
- Remove duplicate records.
- Resolve conflicting Order IDs using financial consistency checks.
- Correct invalid quantity values using transaction calculations.
- Reconcile Sales Amount and Profit values.
- Standardize dates and categorical values.
- Identify and retain missing information.
- Export the cleaned dataset for Exploratory Data Analysis (EDA).

## 2. Dataset Loading and Preparation

The original raw dataset was loaded into Pandas, and a working copy was created to preserve the source data.

Column names were standardized into lowercase snake_case format. Numeric columns containing currency symbols and text formatting were converted into appropriate numeric data types.

## 3. Duplicate Removal and Order ID Resolution

The inspection process identified two types of duplicates:

- Exact duplicate records.
- Conflicting records sharing the same Order ID.

### Actions Taken
- Removed 25 exact duplicate records.
- Evaluated conflicting Order IDs using financial consistency:
  Profit = Sales Amount - Cost Amount.
- Removed 15 mismatched records from 15 conflicting Order ID groups.
- Retained the financially consistent records.

No duplicate Order IDs remained after this process.

## 4. Quantity and Financial Reconciliation

### Quantity Correction

A total of 18 records contained negative or zero quantity values.

These values were corrected using the transaction formula:

Quantity = (Sales Amount - Shipping Fee) /
           (Unit Price × (1 - Discount Percentage / 100))

All 18 quantity records were successfully corrected.

### Sales Reconciliation

Expected Sales Amount was calculated using:

Sales Amount = Quantity × Unit Price × (1 - Discount / 100)
               + Shipping Fee

20 mismatched Sales Amount records were corrected using the calculated values.

### Profit Reconciliation

Expected Profit was calculated using:

Profit = Sales Amount - Cost Amount

12 mismatched Profit records were corrected.

Original Cost Amount values were retained during reconciliation.

## 5. Data Standardization and Missing Value Treatment

### Date Standardization
- Converted Order Date values into a consistent datetime format.
- Invalid dates were identified and retained as missing values.
- Added a flag to identify invalid date records.

### Categorical Standardization

Standardized inconsistent labels across:
- Category
- Payment Method
- Sales Channel
- Order Status

Leading and trailing whitespace was removed from categorical columns.

### Missing Value Treatment

Missing values were retained where reliable source information was unavailable.

No arbitrary mean, median, or mode imputation was performed.

Invalid Age, Discount Percentage, and Product Rating values were flagged and converted into missing values.

## 6. Final Dataset and Quality Verification

The cleaned dataset was exported as:

`retail_sales_cleaned.csv`

### Final Dataset Summary

| Metric | Result |
|---|---:|
| Original Records | 5,040 |
| Final Records | 5,000 |
| Original Columns | 24 |
| Final Columns | 28 |
| Exact Duplicates Removed | 25 |
| Conflicting Duplicate Records Removed | 15 |
| Quantity Corrections | 18 |
| Sales Corrections | 20 |
| Profit Corrections | 12 |
| Remaining Duplicate Order IDs | 0 |
| Remaining Profit Mismatches | 0 |

### Post-Cleaning Verification

The exported dataset was reloaded and checked for:
- Duplicate records and Order IDs.
- Appropriate data types.
- Missing values.
- Invalid numerical values.
- Financial consistency.
- Date validity.
- Categorical standardization.

### Conclusion

The dataset was cleaned, standardized, and reconciled using predefined business rules.

The final dataset contains 5,000 records and 28 columns, including four additional data-quality flags.

Missing values and invalid dates without reliable replacement information were retained for appropriate handling during analysis.

The cleaned dataset is ready for Exploratory Data Analysis (EDA).


In [1]:
# Load the original dataset and convert financial and quantity columns into numeric format.

import pandas as pd
import numpy as np

raw_df = pd.read_csv("../data/raw/retail_sales_raw.csv")

df = raw_df.copy()

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

numeric_columns = [
    "quantity", "unit_price", "discount_pct",
    "shipping_fee", "sales_amount",
    "cost_amount", "profit", "age",
    "delivery_days", "product_rating"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col].astype("string")
        .str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce"
    )

print("Original Dataset:", raw_df.shape)
print("Working Dataset:", df.shape)

Original Dataset: (5040, 24)
Working Dataset: (5040, 24)


In [2]:
# Remove exact duplicates and resolve conflicting Order IDs using financial consistency.

exact_duplicates = df.duplicated().sum()
df = df.drop_duplicates().copy()

df["profit_match"] = np.isclose(
    df["profit"],
    df["sales_amount"] - df["cost_amount"],
    atol=0.01,
    equal_nan=False
)

duplicate_mask = df["order_id"].duplicated(keep=False)
duplicate_groups = df.loc[duplicate_mask].groupby("order_id")

records_to_remove = []

for order_id, group in duplicate_groups:
    if group["profit_match"].sum() == 1:
        records_to_remove.extend(
            group.index[~group["profit_match"]]
        )

df = df.drop(index=records_to_remove).copy()

df.drop(columns="profit_match", inplace=True)

print("Exact duplicates removed:", exact_duplicates)
print("Conflicting duplicate records removed:", len(records_to_remove))
print("Remaining duplicate Order IDs:", df["order_id"].duplicated().sum())
print("Records after cleaning:", len(df))

Exact duplicates removed: 25
Conflicting duplicate records removed: 15
Remaining duplicate Order IDs: 0
Records after cleaning: 5000


/var/folders/01/sgd5rw450c32b5c1ts5pl0vm0000gn/T/ipykernel_59212/559381030.py:26: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns="profit_match", inplace=True)


In [3]:
# Correct negative and zero quantities using sales, unit price, discount, and shipping values.

quantity_mask = df["quantity"].le(0)

calculated_quantity = (
    (df["sales_amount"] - df["shipping_fee"])
    / (
        df["unit_price"]
        * (1 - df["discount_pct"] / 100)
    )
)

rounded_quantity = calculated_quantity.round()

valid_quantity = (
    quantity_mask
    & calculated_quantity.gt(0)
    & np.isclose(calculated_quantity, rounded_quantity, atol=0.01)
)

corrected_quantity_count = int(valid_quantity.sum())

df.loc[valid_quantity, "quantity"] = rounded_quantity[valid_quantity]

print("Negative/Zero quantity records:", int(quantity_mask.sum()))
print("Quantities successfully corrected:", corrected_quantity_count)
print("Unresolved quantity records:", int(df["quantity"].le(0).sum()))

Negative/Zero quantity records: 18
Quantities successfully corrected: 18
Unresolved quantity records: 0


In [4]:
# Reconcile sales and profit values using validated transaction calculations.

expected_sales = (
    df["quantity"] * df["unit_price"]
    * (1 - df["discount_pct"] / 100)
    + df["shipping_fee"]
)

valid_financial_records = (
    df["quantity"].gt(0)
    & df["unit_price"].gt(0)
    & df["discount_pct"].between(0, 100)
    & df["shipping_fee"].ge(0)
)

sales_mismatch = (
    valid_financial_records
    & ~np.isclose(df["sales_amount"], expected_sales, atol=0.02)
)

sales_mismatch_count = int(sales_mismatch.sum())
df.loc[sales_mismatch, "sales_amount"] = expected_sales[sales_mismatch]

profit_mismatch = (
    df["profit"].notna()
    & df["sales_amount"].notna()
    & df["cost_amount"].notna()
    & ~np.isclose(
        df["profit"],
        df["sales_amount"] - df["cost_amount"],
        atol=0.01
    )
)

profit_mismatch_count = int(profit_mismatch.sum())
df.loc[profit_mismatch, "profit"] = (
    df.loc[profit_mismatch, "sales_amount"]
    - df.loc[profit_mismatch, "cost_amount"]
)

print("Sales mismatches corrected:", sales_mismatch_count)
print("Profit mismatches corrected:", profit_mismatch_count)

Sales mismatches corrected: 20
Profit mismatches corrected: 12


In [5]:
# Standardize order dates and categorical values using the identified inspection findings.

df["order_date"] = pd.to_datetime(
    df["order_date"], format="mixed", errors="coerce"
)

categorical_columns = [
    "customer_segment", "gender", "city", "state",
    "category", "sub_category", "payment_method",
    "sales_channel", "order_status", "product_name"
]

for col in categorical_columns:
    df[col] = df[col].str.strip()

mappings = {
    "category": {
        "electronics": "Electronics",
        "Home and Kitchen": "Home & Kitchen"
    },
    "payment_method": {
        "Credit card": "Credit Card",
        "upi": "UPI",
        "NetBanking": "Net Banking",
        "COD": "Cash on Delivery"
    },
    "sales_channel": {
        "mobile app": "Mobile App",
        "website": "Website",
        "Market Place": "Marketplace"
    },
    "order_status": {
        "delivered": "Delivered",
        "CANCELLED": "Cancelled",
        "Return": "Returned"
    }
}

for col, mapping in mappings.items():
    df[col] = df[col].replace(mapping)

In [6]:
# Finalize data quality flags, preserve genuine missing values, and export the cleaned dataset.

df["invalid_date_flag"] = df["order_date"].isna()

df["invalid_age_flag"] = (
    df["age"].notna() & ~df["age"].between(13, 100)
)

df["invalid_discount_flag"] = (
    df["discount_pct"].notna()
    & ~df["discount_pct"].between(0, 100)
)

df["invalid_rating_flag"] = (
    df["product_rating"].notna()
    & ~df["product_rating"].between(1, 5)
)

df.loc[df["invalid_age_flag"], "age"] = np.nan
df.loc[df["invalid_discount_flag"], "discount_pct"] = np.nan
df.loc[df["invalid_rating_flag"], "product_rating"] = np.nan

df.to_csv(
    "../data/processed/retail_sales_cleaned.csv",
    index=False
)

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Original Records",
        "Final Records",
        "Exact Duplicates Removed",
        "Conflicting Duplicate Records Removed",
        "Quantity Corrections",
        "Sales Corrections",
        "Profit Corrections",
        "Invalid Dates",
        "Remaining Missing Values"
    ],
    "Value": [
        len(raw_df),
        len(df),
        exact_duplicates,
        len(records_to_remove),
        corrected_quantity_count,
        sales_mismatch_count,
        profit_mismatch_count,
        int(df["invalid_date_flag"].sum()),
        int(df.isna().sum().sum())
    ]
})

print("Cleaned dataset saved successfully!")
display(cleaning_summary)

Cleaned dataset saved successfully!


,Metric,Value
0,Original Records,5040
1,Final Records,5000
2,Exact Duplicates Removed,25
3,Conflicting Duplicate Records Removed,15
4,Quantity Corrections,18
5,Sales Corrections,20
6,Profit Corrections,12
7,Invalid Dates,30
8,Remaining Missing Values,3285


In [8]:
# Verify the exported dataset for duplicates, data types, missing values, and financial consistency.

final_df = pd.read_csv("../data/processed/retail_sales_cleaned.csv")

verification = {
    "Exported Rows": len(final_df),
    "Exported Columns": len(final_df.columns),
    "Duplicate Rows": int(final_df.duplicated().sum()),
    "Duplicate Order IDs": int(final_df["order_id"].duplicated().sum()),
    "Invalid Quantity": int((final_df["quantity"] <= 0).sum()),
    "Invalid Discount": int(
        ((final_df["discount_pct"] < 0) |
         (final_df["discount_pct"] > 100)).sum()
    ),
    "Invalid Age": int(
        ((final_df["age"] < 13) |
         (final_df["age"] > 100)).sum()
    ),
    "Invalid Rating": int(
        ((final_df["product_rating"] < 1) |
         (final_df["product_rating"] > 5)).sum()
    ),
    "Profit Mismatches": int(
        (~np.isclose(
            final_df["profit"],
            final_df["sales_amount"] - final_df["cost_amount"],
            atol=0.01,
            equal_nan=False
        )).sum()
    ),
    "Missing Values": int(final_df.isna().sum().sum())
}

verification_report = pd.DataFrame(
    verification.items(),
    columns=["Verification Check", "Result"]
)

display(verification_report)

,Verification Check,Result
0,Exported Rows,5000
1,Exported Columns,28
2,Duplicate Rows,0
3,Duplicate Order IDs,0
4,Invalid Quantity,0
5,Invalid Discount,0
6,Invalid Age,0
7,Invalid Rating,0
8,Profit Mismatches,0
9,Missing Values,3285


In [9]:
# Display the final dataset columns and their data types.
display(pd.DataFrame({
    "Column": final_df.columns,
    "Data Type": final_df.dtypes.astype(str).values
}))

,Column,Data Type
0,order_id,str
1,order_date,str
2,customer_id,str
3,customer_segment,str
4,gender,str
5,age,float64
6,city,str
7,state,str
8,product_id,str
9,category,str


In [11]:
# Perform a complete quality inspection of the exported cleaned dataset.

final_df = pd.read_csv("../data/processed/retail_sales_cleaned.csv")

# 1. Dataset overview
print("DATASET OVERVIEW")
print("Shape:", final_df.shape)
print("Memory Usage:", round(final_df.memory_usage(deep=True).sum() / 1024, 2), "KB")

# 2. Data types
print("\nDATA TYPES")
display(final_df.dtypes.astype(str).rename("Data Type").to_frame())

# 3. Missing values
print("\nMISSING VALUES")
missing_report = pd.DataFrame({
    "Missing Count": final_df.isna().sum(),
    "Missing Percentage": (
        final_df.isna().mean() * 100
    ).round(2)
})
display(missing_report[missing_report["Missing Count"] > 0])

# 4. Duplicate records
print("\nDUPLICATE CHECK")
print("Exact Duplicate Rows:", final_df.duplicated().sum())
print("Duplicate Order IDs:", final_df["order_id"].duplicated().sum())

# 5. Numerical statistics
print("\nNUMERICAL SUMMARY")
display(final_df.describe().T)

# 6. Categorical summary
print("\nCATEGORICAL SUMMARY")
categorical_columns = final_df.select_dtypes(
    include=["object", "str"]
).columns

display(final_df[categorical_columns].nunique().to_frame("Unique Values"))

# 7. Date range
print("\nDATE SUMMARY")
final_df["order_date"] = pd.to_datetime(
    final_df["order_date"], errors="coerce"
)
print("Earliest Order Date:", final_df["order_date"].min())
print("Latest Order Date:", final_df["order_date"].max())
print("Invalid Dates:", final_df["order_date"].isna().sum())

# 8. Business rule validation
print("\nBUSINESS VALIDATION")
print("Invalid Quantity:", (final_df["quantity"] <= 0).sum())
print("Invalid Discount:", (
    ~final_df["discount_pct"].between(0, 100)
    & final_df["discount_pct"].notna()
).sum())
print("Invalid Age:", (
    ~final_df["age"].between(13, 100)
    & final_df["age"].notna()
).sum())
print("Invalid Rating:", (
    ~final_df["product_rating"].between(1, 5)
    & final_df["product_rating"].notna()
).sum())

valid_profit = final_df[
    ["profit", "sales_amount", "cost_amount"]
].notna().all(axis=1)

profit_mismatch = ~np.isclose(
    final_df.loc[valid_profit, "profit"],
    final_df.loc[valid_profit, "sales_amount"]
    - final_df.loc[valid_profit, "cost_amount"],
    atol=0.01
)

print("Profit Mismatches:", profit_mismatch.sum())

print("\nORDER STATUS DISTRIBUTION")
display(final_df["order_status"].value_counts(dropna=False).to_frame("Records"))

DATASET OVERVIEW
Shape: (5000, 28)
Memory Usage: 4373.96 KB

DATA TYPES


,Data Type
order_id,str
order_date,str
customer_id,str
customer_segment,str
gender,str
age,float64
city,str
state,str
product_id,str
category,str



MISSING VALUES


,Missing Count,Missing Percentage
order_date,30,0.60
customer_segment,60,1.20
age,72,1.44
city,90,1.80
discount_pct,106,2.12
payment_method,125,2.50
delivery_days,1392,27.84
product_rating,1410,28.20



DUPLICATE CHECK
Exact Duplicate Rows: 0
Duplicate Order IDs: 0

NUMERICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
age,4928.0,41.266640,13.831950,18.00,29.000,41.000,53.0000,65.00
quantity,5000.0,2.379400,1.413455,1.00,1.000,2.000,4.0000,5.00
unit_price,5000.0,2251.199050,1889.604496,539.10,809.100,1619.100,2969.1000,7348.95
discount_pct,4894.0,10.524111,8.425879,0.00,5.000,10.000,15.0000,25.00
shipping_fee,5000.0,39.997200,38.493937,0.00,0.000,40.000,80.0000,99.00
sales_amount,5000.0,4848.109541,5479.388559,404.33,1438.200,2876.000,5938.2000,36784.75
cost_amount,5000.0,3499.679934,4048.340495,259.52,1001.695,2017.180,4179.4975,29805.85
profit,5000.0,1348.429606,1809.063356,-1662.28,320.650,734.075,1581.5350,17841.50
delivery_days,3608.0,5.002494,2.552090,1.00,3.000,5.000,7.0000,9.00
product_rating,3590.0,3.001671,1.413424,1.00,2.000,3.000,4.0000,5.00



CATEGORICAL SUMMARY


,Unique Values
order_id,5000
order_date,365
customer_id,1675
customer_segment,3
gender,4
city,14
state,12
product_id,12
category,6
sub_category,11



DATE SUMMARY
Earliest Order Date: 2025-01-01 00:00:00
Latest Order Date: 2025-12-31 00:00:00
Invalid Dates: 30

BUSINESS VALIDATION
Invalid Quantity: 0
Invalid Discount: 0
Invalid Age: 0
Invalid Rating: 0
Profit Mismatches: 0

ORDER STATUS DISTRIBUTION


,Records
order_status,
Delivered,3444
Shipped,532
Processing,380
Cancelled,350
Returned,294


In [13]:
print(final_df)

        order_id order_date customer_id customer_segment      gender   age  \
0     ORD-004477 2025-03-12  CUST-00470          Loyalty  Non-binary  44.0   
1     ORD-001664 2025-03-04  CUST-00961        Returning        Male  20.0   
2     ORD-000630 2025-03-17  CUST-01357          Loyalty      Female  64.0   
3     ORD-000101 2025-06-07  CUST-01050          Loyalty  Non-binary  41.0   
4     ORD-001509 2025-03-30  CUST-00819              New  Non-binary  26.0   
...          ...        ...         ...              ...         ...   ...   
4995  ORD-001318 2025-02-17  CUST-01666              New      Female  65.0   
4996  ORD-002284 2025-06-29  CUST-01616          Loyalty      Female  48.0   
4997  ORD-002005 2025-02-04  CUST-00933          Loyalty  Non-binary  61.0   
4998  ORD-003669 2025-05-10  CUST-01481        Returning      Female  30.0   
4999  ORD-000608 2025-08-12  CUST-00773        Returning      Female  56.0   

            city           state product_id        category  ..